# ตัวอย่างประกอบการสอน: Graphical vs Simplex Methods

**แนวคิดหลักของสไลด์ (หน้า 54):** *"Simplex เดินจาก 'จุดมุม' หนึ่งไปยังจุดมุมถัดไปที่ทำให้ Objective ดีขึ้น จนไม่มีทางดีขึ้นต่อ"*

ไฟล์นี้เอา**โจทย์เดียวกัน** (Example 1 จากสไลด์ Simplex) มาแก้ **2 วิธีพร้อมกัน** แล้วพิสูจน์ให้เห็นภาพจริงว่า:

> ทุกครั้งที่ Simplex ทำ pivot 1 ครั้ง = มันกำลัง **"กระโดด" จากจุดมุมหนึ่งไปอีกจุดมุมหนึ่งบนกราฟเดียวกับที่ Graphical Method หาเจอ**

**โจทย์:** Maximize $Z=3x_1+5x_2$

ข้อจำกัด: $x_1\le4,\quad 2x_2\le12,\quad 3x_1+2x_2\le18,\quad x_1,x_2\ge0$


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.precision', 2)


## ส่วนที่ 1 — Graphical Method: หาจุดมุมทั้งหมดก่อน (ภาพ "ก่อน" ที่ยังไม่รู้คำตอบ)

วิธีกราฟจะหาจุดมุม (corner point) **ทุกจุด** ของ feasible region แล้วเทียบค่า Z — เป็นการมองภาพรวมทั้งหมดของปัญหา
ก่อนจะรู้ว่าคำตอบคือจุดไหน


In [ ]:
c = [3, 5]
A = [[1, 0], [0, 2], [3, 2]]
b = [4, 12, 18]


def find_corners(A, b):
    """หาจุดมุมทั้งหมดของ feasible region จากจุดตัดของทุกคู่เส้น (รวมแกน x=0, y=0)"""
    lines = list(A) + [[1, 0], [0, 1]]
    rhs = list(b) + [0, 0]
    corners = []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            Amat = np.array([lines[i], lines[j]])
            bvec = np.array([rhs[i], rhs[j]])
            if abs(np.linalg.det(Amat)) < 1e-9:
                continue
            pt = np.linalg.solve(Amat, bvec)
            if pt[0] >= -1e-6 and pt[1] >= -1e-6:
                if all(a1 * pt[0] + a2 * pt[1] <= bi + 1e-6 for (a1, a2), bi in zip(A, b)):
                    corners.append(tuple(np.round(pt, 4)))
    return sorted(set(corners))


corners = find_corners(A, b)
corner_table = pd.DataFrame({
    "จุดมุม": [f"({x:.0f},{y:.0f})" for x, y in corners],
    "Z=3x1+5x2": [c[0] * x + c[1] * y for x, y in corners],
})
corner_table = corner_table.sort_values("Z=3x1+5x2").reset_index(drop=True)
print("=== ตารางจุดมุมทั้งหมด (Graphical Method) ===")
print(corner_table)
best_row = corner_table.loc[corner_table["Z=3x1+5x2"].idxmax()]
print(f"\n>>> Optimal: {best_row['จุดมุม']}  Z={best_row['Z=3x1+5x2']:.0f}")


## ส่วนที่ 2 — Simplex Method: ดู "ก่อน/หลัง" ทีละ pivot

Simplex **ไม่รู้จักจุดมุมทั้งหมดล่วงหน้าเหมือน Graphical** — มันเริ่มจากจุดเดียว (origin) แล้วค่อยๆ **เดินทีละก้าว**
ไปยังจุดมุมที่ดีกว่า จนกว่าจะเจอจุดที่ดีที่สุด เราจะพิมพ์สถานะ **ก่อน pivot** และ **หลัง pivot** ของทุกรอบ


In [ ]:
def simplex_maximize(c, A, b, var_names=None):
    c = np.array(c, dtype=float); A = np.array(A, dtype=float); b = np.array(b, dtype=float)
    n_vars = len(c); n_cons = len(b)
    if var_names is None:
        var_names = [f"x{i+1}" for i in range(n_vars)]
    slack_names = [f"S{i+1}" for i in range(n_cons)]
    all_names = var_names + slack_names + ["RHS"]

    tableau = np.zeros((n_cons + 1, n_vars + n_cons + 1))
    tableau[:n_cons, :n_vars] = A
    tableau[:n_cons, n_vars:n_vars + n_cons] = np.eye(n_cons)
    tableau[:n_cons, -1] = b
    tableau[-1, :n_vars] = -c
    basic = slack_names.copy()

    def get_point(tableau, basic):
        sol = {name: 0.0 for name in var_names}
        for i, name in enumerate(basic):
            if name in sol:
                sol[name] = tableau[i, -1]
        return sol, tableau[-1, -1]

    steps = []
    pt, z = get_point(tableau, basic)
    steps.append({"label": "จุดเริ่มต้น (origin)", "point": pt, "Z": z, "tableau": tableau.copy(), "basic": basic.copy()})

    it = 0
    while True:
        z_row = tableau[-1, :-1]
        if np.all(z_row >= -1e-9):
            break
        entering = np.argmin(z_row)
        col = tableau[:n_cons, entering]
        rhs = tableau[:n_cons, -1]
        ratios = np.where(col > 1e-9, rhs / np.where(col > 1e-9, col, 1), np.inf)
        leaving = np.argmin(ratios)
        pivot_val = tableau[leaving, entering]

        it += 1
        entering_name, leaving_name = all_names[entering], basic[leaving]

        before_pt, before_z = get_point(tableau, basic)

        tableau[leaving, :] /= pivot_val
        for r in range(len(tableau)):
            if r != leaving:
                tableau[r, :] -= tableau[r, entering] * tableau[leaving, :]
        basic[leaving] = entering_name

        after_pt, after_z = get_point(tableau, basic)

        print(f"### Pivot {it}: {entering_name} เข้าแทน {leaving_name}")
        print(f"  ก่อน pivot : point={tuple(round(float(v),1) for v in before_pt.values())}  Z={before_z:.1f}")
        print(f"  หลัง pivot : point={tuple(round(float(v),1) for v in after_pt.values())}  Z={after_z:.1f}")
        print(f"  --> Z {'เพิ่มขึ้น' if after_z > before_z else 'เท่าเดิม'} จาก {before_z:.1f} เป็น {after_z:.1f}\n")

        steps.append({"label": f"หลัง pivot {it}: {entering_name} เข้าแทน {leaving_name}",
                      "point": after_pt, "Z": after_z, "tableau": tableau.copy(), "basic": basic.copy()})

    return steps


steps = simplex_maximize(c, A, b, var_names=["x1", "x2"])
print(f"จำนวน pivot ทั้งหมด: {len(steps)-1} ครั้ง (จาก {len(corners)} จุดมุมทั้งหมดในกราฟ)")


## ส่วนที่ 3 — รวมภาพ: Simplex เดินบนกราฟเดียวกับ Graphical Method

นำเส้นทางที่ Simplex เดิน (จาก `steps`) มาซ้อนทับบนกราฟ feasible region เดียวกับ Graphical Method
เพื่อพิสูจน์ให้เห็นว่ามันคือกระบวนการเดียวกัน เพียงแค่มองจากมุมต่างกัน


In [ ]:
def plot_teaching_demo(A, b, c, corners, steps, xr=(0, 8), yr=(0, 8)):
    xs = np.linspace(*xr, 300)
    ys = np.linspace(*yr, 300)
    X, Y = np.meshgrid(xs, ys)
    feasible = np.ones_like(X, dtype=bool)
    for (a1, a2), bi in zip(A, b):
        feasible &= (a1 * X + a2 * Y <= bi + 1e-9)
    feasible &= (X >= 0) & (Y >= 0)

    fig = go.Figure()

    # feasible region
    fig.add_trace(go.Contour(
        x=xs, y=ys, z=feasible.astype(int), showscale=False,
        contours={"start": 0.5, "end": 1, "size": 1},
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(80,160,120,0.30)"]],
        line={"width": 0}, hoverinfo="skip", name="Feasible region",
    ))

    # constraint lines
    for (a1, a2), bi in zip(A, b):
        if a2 != 0:
            yy = (bi - a1 * xs) / a2
            fig.add_trace(go.Scatter(x=xs, y=yy, mode="lines", line={"dash": "dot", "color": "gray"},
                                      name=f"{a1:g}x1+{a2:g}x2={bi:g}"))
        else:
            fig.add_trace(go.Scatter(x=[bi / a1] * 2, y=list(yr), mode="lines",
                                      line={"dash": "dot", "color": "gray"}, name=f"{a1:g}x1={bi:g}"))

    # ALL corner points (สีเทา = Graphical Method หาเจอทั้งหมด)
    cxs = [p[0] for p in corners]
    cys = [p[1] for p in corners]
    czs = [c[0] * p[0] + c[1] * p[1] for p in corners]
    fig.add_trace(go.Scatter(
        x=cxs, y=cys, mode="markers+text", marker={"size": 10, "color": "lightgray", "line": {"width": 1, "color": "gray"}},
        text=[f"Z={z:.0f}" for z in czs], textposition="bottom center",
        name="ทุกจุดมุม (Graphical เจอหมด)",
    ))

    # Simplex path (สีแดง = เส้นทางที่ Simplex เดินจริง)
    path_x = [s["point"]["x1"] for s in steps]
    path_y = [s["point"]["x2"] for s in steps]
    path_z = [s["Z"] for s in steps]
    fig.add_trace(go.Scatter(
        x=path_x, y=path_y, mode="lines+markers",
        line={"color": "red", "width": 3}, marker={"size": 14, "color": "red", "symbol": "star"},
        text=[f"{s['label']}<br>Z={s['Z']:.0f}" for s in steps], textposition="top center",
        name="เส้นทางที่ Simplex เดินจริง",
    ))

    fig.update_layout(
        title="Graphical Method (จุดเทา=หาทุกจุด) vs Simplex Method (เส้นแดง=เดินทีละก้าว)",
        xaxis_title="x1", yaxis_title="x2",
        xaxis={"range": xr}, yaxis={"range": yr},
        width=700, height=600,
    )
    fig.show()


plot_teaching_demo(A, b, c, corners, steps)


## ส่วนที่ 4 — สรุปบทเรียน

| จุดมุมทั้งหมด (Graphical) | Z | Simplex เดินผ่านไหม? |
|---|---|---|
| (0,0) | 0 | ✅ จุดเริ่มต้น |
| (4,0) | 12 | ❌ ไม่เดินผ่าน |
| (4,3) | 27 | ❌ ไม่เดินผ่าน |
| (0,6) | 30 | ✅ หลัง pivot 1 |
| (2,6) | 36 | ✅ หลัง pivot 2 (Optimal) |

**ข้อสรุปสำคัญสำหรับสอนนักเรียน:**

1. **Graphical Method** ต้องหาจุดมุม**ทุกจุด** (5 จุดในที่นี้) แล้วค่อยเทียบ — ทำงานแบบ "มองภาพรวมทั้งหมดก่อนตัดสินใจ"
2. **Simplex Method** เดินผ่านแค่ **3 จุดจาก 5 จุด** (60% ของจุดมุมทั้งหมด) — ทำงานแบบ "เดินเฉพาะเส้นทางที่ดีขึ้นเรื่อยๆ ไม่เสียเวลาไปดูจุดที่ไม่มีทางดีกว่า"
3. ค่า Z ที่ Simplex เจอ **เพิ่มขึ้นทุกครั้ง** ($0\to30\to36$) ไม่เคยลดลง — นี่คือเหตุผลที่ Simplex รับประกันว่าจะเจอ optimal โดยไม่ต้องย้อนกลับ
4. ทั้งสองวิธี**ได้คำตอบเดียวกันเป๊ะ** ($x_1=2,x_2=6,Z=36$) เพราะมันคือปัญหาเดียวกัน แค่มองจากคนละมุม

> 💡 **สำหรับผู้สอน:** ยิ่งปัญหามีตัวแปร/ข้อจำกัดมากขึ้น จำนวนจุดมุมจะเพิ่มแบบ exponential (Graphical วาดไม่ได้เกิน 2-3 ตัวแปร)
> แต่ Simplex ยังคง "เดินเฉพาะเส้นทางที่ดีขึ้น" ได้เสมอ — นี่คือเหตุผลที่ Simplex ใช้แก้ปัญหาจริงที่มีตัวแปรหลักพันหลักหมื่นได้ ในขณะที่ Graphical ทำไม่ได้เลย
